In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import xgboost as xgb
import pickle

trans = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
identity = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')

df = trans.merge(identity, on='TransactionID', how='left')

missing = (df.isnull().sum() / len(df) * 100)
cols_to_drop = missing[missing > 90].index.tolist()
df = df.drop(columns=cols_to_drop)
numerical_cols = df.select_dtypes(include=[np.number]).columns
df[numerical_cols] = df[numerical_cols].fillna(df[numerical_cols].median())
categorical_cols = df.select_dtypes(include=['object']).columns
df[categorical_cols] = df[categorical_cols].fillna('Unknown')
df = df.copy()

df['hour'] = (df['TransactionDT'] // 3600) % 24
df['day'] = (df['TransactionDT'] // (3600 * 24)) % 7
df['is_late_night'] = df['hour'].apply(lambda x: 1 if 1 <= x <= 9 else 0)
df['log_amount'] = np.log1p(df['TransactionAmt'])
df['is_card_probe'] = (df['TransactionAmt'] < 1).astype(int)
df['is_threshold_avoid'] = ((df['TransactionAmt'] >= 500) & (df['TransactionAmt'] < 1000)).astype(int)
df['has_identity'] = df['DeviceType'].apply(lambda x: 0 if x == 'Unknown' else 1)
df['is_mobile'] = df['DeviceType'].apply(lambda x: 1 if x == 'mobile' else 0)

card_counts = df.groupby('card1')['TransactionAmt'].count().reset_index()
card_counts.columns = ['card1', 'card1_count']
df = df.merge(card_counts, on='card1', how='left')

card_mean = df.groupby('card1')['TransactionAmt'].mean().reset_index()
card_mean.columns = ['card1', 'card1_mean_amt']
df = df.merge(card_mean, on='card1', how='left')

df['amt_deviation'] = abs(df['TransactionAmt'] - df['card1_mean_amt'])
df['amt_deviation_ratio'] = df['amt_deviation'] / (df['card1_mean_amt'] + 1)

free_emails = ['gmail.com', 'yahoo.com', 'hotmail.com', 'outlook.com', 'aol.com', 'mail.com']
df['is_free_email'] = df['P_emaildomain'].apply(lambda x: 1 if x in free_emails else 0)
df['email_match'] = (df['P_emaildomain'] == df['R_emaildomain']).astype(int)

def fraud_rate_encode(df, col):
    fraud_rates = df.groupby(col)['isFraud'].mean()
    return df[col].map(fraud_rates)

df['product_fraud_rate'] = fraud_rate_encode(df, 'ProductCD')
df['card4_fraud_rate'] = fraud_rate_encode(df, 'card4')
df['card6_fraud_rate'] = fraud_rate_encode(df, 'card6')

drop_cols = ['TransactionID', 'isFraud', 'TransactionDT',
             'ProductCD', 'card4', 'card6',
             'P_emaildomain', 'R_emaildomain', 'DeviceType']
drop_cols = [c for c in drop_cols if c in df.columns]
string_cols = df.drop(columns=drop_cols).select_dtypes(include=['object']).columns.tolist()

X = df.drop(columns=drop_cols + string_cols)
y = df['isFraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# retrain model
scale = (y_train == 0).sum() / (y_train == 1).sum()
best_model = xgb.XGBClassifier(
    n_estimators=500, max_depth=8, learning_rate=0.2,
    subsample=0.9, colsample_bytree=0.8,
    scale_pos_weight=scale, random_state=42,
    tree_method='hist', device='cuda', eval_metric='aucpr'
)
best_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=100)

print("Ready!")

[0]	validation_0-aucpr:0.37366
[100]	validation_0-aucpr:0.72615
[200]	validation_0-aucpr:0.77933
[300]	validation_0-aucpr:0.79934
[400]	validation_0-aucpr:0.81476
[499]	validation_0-aucpr:0.82392
Ready!


In [2]:
# load competition test data
test_trans = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv')
test_identity = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv')

test_df = test_trans.merge(test_identity, on='TransactionID', how='left')

# same missing value treatment
missing_test = (test_df.isnull().sum() / len(test_df) * 100)
cols_to_drop_test = missing_test[missing_test > 90].index.tolist()
test_df = test_df.drop(columns=cols_to_drop_test)

numerical_cols_test = test_df.select_dtypes(include=[np.number]).columns
test_df[numerical_cols_test] = test_df[numerical_cols_test].fillna(test_df[numerical_cols_test].median())
categorical_cols_test = test_df.select_dtypes(include=['object']).columns
test_df[categorical_cols_test] = test_df[categorical_cols_test].fillna('Unknown')
test_df = test_df.copy()

# same features
test_df['hour'] = (test_df['TransactionDT'] // 3600) % 24
test_df['day'] = (test_df['TransactionDT'] // (3600 * 24)) % 7
test_df['is_late_night'] = test_df['hour'].apply(lambda x: 1 if 1 <= x <= 9 else 0)
test_df['log_amount'] = np.log1p(test_df['TransactionAmt'])
test_df['is_card_probe'] = (test_df['TransactionAmt'] < 1).astype(int)
test_df['is_threshold_avoid'] = ((test_df['TransactionAmt'] >= 500) & (test_df['TransactionAmt'] < 1000)).astype(int)
test_df['has_identity'] = test_df['DeviceType'].apply(lambda x: 0 if x == 'Unknown' else 1)
test_df['is_mobile'] = test_df['DeviceType'].apply(lambda x: 1 if x == 'mobile' else 0)

card_counts_test = test_df.groupby('card1')['TransactionAmt'].count().reset_index()
card_counts_test.columns = ['card1', 'card1_count']
test_df = test_df.merge(card_counts_test, on='card1', how='left')

card_mean_test = test_df.groupby('card1')['TransactionAmt'].mean().reset_index()
card_mean_test.columns = ['card1', 'card1_mean_amt']
test_df = test_df.merge(card_mean_test, on='card1', how='left')

test_df['amt_deviation'] = abs(test_df['TransactionAmt'] - test_df['card1_mean_amt'])
test_df['amt_deviation_ratio'] = test_df['amt_deviation'] / (test_df['card1_mean_amt'] + 1)

free_emails = ['gmail.com', 'yahoo.com', 'hotmail.com', 'outlook.com', 'aol.com', 'mail.com']
test_df['is_free_email'] = test_df['P_emaildomain'].apply(lambda x: 1 if x in free_emails else 0)
test_df['email_match'] = (test_df['P_emaildomain'] == test_df['R_emaildomain']).astype(int)

# use fraud rates learned from training data only
test_df['product_fraud_rate'] = test_df['ProductCD'].map(df.groupby('ProductCD')['isFraud'].mean()).fillna(0)
test_df['card4_fraud_rate'] = test_df['card4'].map(df.groupby('card4')['isFraud'].mean()).fillna(0)
test_df['card6_fraud_rate'] = test_df['card6'].map(df.groupby('card6')['isFraud'].mean()).fillna(0)

# align columns with training
drop_cols_test = ['TransactionID', 'TransactionDT', 'ProductCD', 'card4', 'card6',
                  'P_emaildomain', 'R_emaildomain', 'DeviceType']
drop_cols_test = [c for c in drop_cols_test if c in test_df.columns]
string_cols_test = test_df.drop(columns=drop_cols_test).select_dtypes(include=['object']).columns.tolist()

X_test_final = test_df.drop(columns=drop_cols_test + string_cols_test)

for col in X_train.columns:
    if col not in X_test_final.columns:
        X_test_final[col] = 0

X_test_final = X_test_final[X_train.columns]

# generate predictions
predictions = best_model.predict_proba(X_test_final)[:, 1]

# create submission file
submission = pd.DataFrame({
    'TransactionID': test_trans['TransactionID'],
    'isFraud': predictions
})

submission.to_csv('/kaggle/working/submission.csv', index=False)
print("Submission file created!")
print(f"Shape: {submission.shape}")
print(f"Average fraud probability: {predictions.mean()*100:.2f}%")
print(submission.head())

/usr/local/lib/python3.12/dist-packages/xgboost/core.py:774: UserWarning: [12:46:51] WARNING: /workspace/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Submission file created!
Shape: (506691, 2)
Average fraud probability: 0.89%
   TransactionID       isFraud
0        3663549  1.316986e-06
1        3663550  3.156744e-05
2        3663551  5.228991e-07
3        3663552  9.348440e-06
4        3663553  1.861528e-06
